In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:

# =========================================================
# 0. IMPORTS
# =========================================================
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================================================
# 1. DATASET
# =========================================================
class EchoDataset(Dataset):
    def __init__(self, images, masks):
        self.images = images
        self.masks = masks

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = cv2.imread(self.images[idx])
        img = cv2.resize(img, (128,128))
        img = img / 255.0
        img = np.transpose(img, (2,0,1)).astype(np.float32)

        mask = cv2.imread(self.masks[idx], 0)
        mask = cv2.resize(mask, (128,128))
        mask = (mask > 0).astype(np.float32)
        mask = np.expand_dims(mask, 0)

        return torch.tensor(img), torch.tensor(mask)

# =========================================================
# 2. LOAD DATA
# =========================================================
DATA_PATH = "/kaggle/input/datasets/harshithabhaskaran/echonet-final-dataset-001"
IMG_DIR = os.path.join(DATA_PATH, "images")
MASK_DIR = os.path.join(DATA_PATH, "masks")

images = sorted([os.path.join(IMG_DIR,f) for f in os.listdir(IMG_DIR)])
masks  = sorted([os.path.join(MASK_DIR,f) for f in os.listdir(MASK_DIR)])

train_i, val_i, train_m, val_m = train_test_split(
    images, masks, test_size=0.2, random_state=42
)

train_ds = EchoDataset(train_i, train_m)
val_ds   = EchoDataset(val_i, val_m)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=4, shuffle=False)

print("Train:", len(train_ds), "Val:", len(val_ds))

# =========================================================
# 3. U-NET (FULLY SYMMETRIC - FIXED)
# =========================================================
class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        def block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.ReLU(),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.ReLU()
            )

        # Encoder
        self.enc1 = block(3, 32)
        self.enc2 = block(32, 64)

        self.pool = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = block(64, 128)

        # Decoder
        self.up1 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec1 = block(128, 64)

        self.up2 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.dec2 = block(64, 32)

        self.final = nn.Conv2d(32, 1, 1)

    def forward(self, x):

        e1 = self.enc1(x)                  # 128
        e2 = self.enc2(self.pool(e1))      # 64

        b  = self.bottleneck(self.pool(e2))# 32

        d1 = self.up1(b)                   # 64
        d1 = self.dec1(torch.cat([d1,e2], dim=1))

        d2 = self.up2(d1)                 # 128
        d2 = self.dec2(torch.cat([d2,e1], dim=1))

        return torch.sigmoid(self.final(d2))

model = UNet().to(device)

# =========================================================
# 4. LOSS (MEDICAL STANDARD)
# =========================================================
bce = nn.BCELoss()

def dice_loss(pred, target):
    pred = pred.view(pred.size(0), -1)
    target = target.view(target.size(0), -1)

    inter = (pred * target).sum(dim=1)
    union = pred.sum(dim=1) + target.sum(dim=1)

    dice = (2 * inter + 1e-5) / (union + 1e-5)
    return 1 - dice.mean()

def loss_fn(pred, target):
    return bce(pred, target) + dice_loss(pred, target)

# =========================================================
# 5. OPTIMIZER
# =========================================================
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# =========================================================
# 6. TRAINING LOOP
# =========================================================
EPOCHS = 5

for epoch in range(EPOCHS):

    model.train()
    train_loss = 0

    for img, mask in train_loader:

        img = img.to(device).float()
        mask = mask.to(device).float()

        pred = model(img)

        # safety check
        assert pred.shape == mask.shape

        loss = loss_fn(pred, mask)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # validation
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for img, mask in val_loader:

            img = img.to(device).float()
            mask = mask.to(device).float()

            pred = model(img)

            val_loss += loss_fn(pred, mask).item()

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("Train Loss:", train_loss)
    print("Val Loss:", val_loss)

# =========================================================
# 7. METRICS (DICE + IOU)
# =========================================================
def dice_score(p, g):
    p = (p > 0.5).float()
    return (2*(p*g).sum()) / ((p+g).sum() + 1e-8)

def iou_score(p, g):
    p = (p > 0.5).float()
    inter = (p*g).sum()
    union = p.sum() + g.sum() - inter
    return inter / (union + 1e-8)

dice_list, iou_list = [], []

model.eval()

with torch.no_grad():
    for img, mask in val_loader:
        img = img.to(device).float()
        mask = mask.to(device).float()

        pred = model(img)

        for i in range(img.size(0)):
            dice_list.append(dice_score(pred[i], mask[i]).item())
            iou_list.append(iou_score(pred[i], mask[i]).item())

print("\n================ FINAL RESULTS ================")
print("Dice Score:", np.mean(dice_list))
print("IoU Score:", np.mean(iou_list))

In [ ]:
import matplotlib.pyplot as plt

train_loss = [1362.75, 780.99, 702.19, 658.71, 630.28]
val_loss   = [204.39, 177.28, 165.33, 163.84, 157.45]

epochs = range(1, 6)

plt.figure()
plt.plot(epochs, train_loss, label="Train Loss")
plt.plot(epochs, val_loss, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

dice = 0.9155
iou = 0.8491

metrics = ["Dice Score", "IoU Score"]
values = [dice, iou]

plt.figure()
plt.bar(metrics, values)
plt.ylim(0, 1)
plt.title("Segmentation Performance Metrics")
plt.grid(axis="y")
plt.show()

In [ ]:
fig, ax = plt.subplots()

ax.plot(epochs, train_loss, label="Train Loss")
ax.plot(epochs, val_loss, label="Val Loss")

ax.set_title("Model Convergence")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
ax.grid()

plt.show()

In [ ]:
import matplotlib.pyplot as plt

model.eval()

img, mask = val_ds[0]
img_in = img.unsqueeze(0).to(device).float()

with torch.no_grad():
    pred = model(img_in)[0][0].cpu().numpy()

img_np = img.permute(1,2,0).numpy()
mask_np = mask[0].numpy()

plt.figure(figsize=(10,3))

plt.subplot(1,3,1)
plt.title("Input Image")
plt.imshow(img_np)

plt.subplot(1,3,2)
plt.title("Ground Truth")
plt.imshow(mask_np, cmap="gray")

plt.subplot(1,3,3)
plt.title("Prediction")
plt.imshow(pred, cmap="gray")

plt.show()

In [ ]:
import numpy as np
import torch

model.eval()

volumes = []

with torch.no_grad():
    for i in range(len(val_ds)):
        img, _ = val_ds[i]

        img = img.unsqueeze(0).to(device).float()
        pred = model(img)[0][0].cpu().numpy()

        # IMPORTANT: threshold for stability (real medical practice)
        pred_bin = (pred > 0.5).astype(np.float32)

        # LV volume proxy = area of mask
        volume = np.sum(pred_bin)
        volumes.append(volume)

volumes = np.array(volumes)

print("Volume signal shape:", volumes.shape)

In [ ]:
from scipy.signal import find_peaks

# ED = peaks (max volume)
ed_peaks, _ = find_peaks(volumes)

# ES = valleys (min volume)
es_peaks, _ = find_peaks(-volumes)

print("ED frames:", ed_peaks)
print("ES frames:", es_peaks)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,4))
plt.plot(volumes, label="LV Volume Curve")

plt.scatter(ed_peaks, volumes[ed_peaks], color="red", label="End Diastole")
plt.scatter(es_peaks, volumes[es_peaks], color="blue", label="End Systole")

plt.title("Cardiac Function from Segmentation")
plt.xlabel("Frame")
plt.ylabel("LV Volume (pixel area)")
plt.legend()
plt.grid()
plt.show()

In [ ]:
ef_list = []

for i in range(min(len(ed_peaks), len(es_peaks))):
    edv = volumes[ed_peaks[i]]
    esv = volumes[es_peaks[i]]

    if edv > 0:
        ef = ((edv - esv) / edv) * 100
        ef_list.append(ef)

ef_list = np.array(ef_list)

print("EF per cycle:", ef_list)
print("Mean EF:", np.mean(ef_list))

In [ ]:
mean_ef = np.mean(ef_list)
std_ef = np.std(ef_list)

print("\n================ CLINICAL INTERPRETATION ================")

if mean_ef < 40:
    print("⚠ SYSTOLIC DYSFUNCTION (LOW EF)")
elif std_ef > 6:
    print("⚠ IRREGULAR CARDIAC CONTRACTION")
elif np.max(volumes) - np.min(volumes) < 10:
    print("⚠ WEAK CONTRACTION (LOW AMPLITUDE)")
else:
    print("✅ NORMAL CARDIAC FUNCTION")

In [ ]:
report = f"""
================ CARDIAC ANALYSIS REPORT ================

Left Ventricle Segmentation: SUCCESSFUL
Dice Score: ~0.91 (from previous training)

Cardiac Function Metrics:
- Mean EF: {np.mean(ef_list):.2f} %
- EF Variability: {np.std(ef_list):.2f}

Findings:
- ED frames detected: {len(ed_peaks)}
- ES frames detected: {len(es_peaks)}

Diagnosis:
"""

if mean_ef < 40:
    report += "Systolic dysfunction suspected (low EF)\n"
elif std_ef > 6:
    report += "Irregular cardiac contraction observed\n"
else:
    report += "Normal cardiac function\n"

print(report)

In [ ]:

import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks

model.eval()

# =========================================================
# 1. EXTRACT LV VOLUME SIGNAL (FROM SEGMENTATION)
# =========================================================
volumes = []

with torch.no_grad():
    for i in range(len(val_ds)):
        img, _ = val_ds[i]

        img = img.unsqueeze(0).to(device).float()
        pred = model(img)[0][0].cpu().numpy()

        # binary mask (stable LV region)
        pred = (pred > 0.5).astype(np.float32)

        volumes.append(np.sum(pred))

volumes = np.array(volumes)

# =========================================================
# 2. SMOOTH SIGNAL (PHYSIOLOGICAL REALISM)
# =========================================================
vol_smooth = gaussian_filter1d(volumes, sigma=3)

# =========================================================
# 3. NORMALIZATION (FOR PEAK DETECTION ONLY)
# =========================================================
vol_norm = (vol_smooth - vol_smooth.min()) / (vol_smooth.max() - vol_smooth.min() + 1e-8)

# =========================================================
# 4. ROBUST CARDIAC CYCLE DETECTION
#    (LIMITS FAKE PEAKS)
# =========================================================
min_distance = max(10, len(vol_norm) // 4)

ed_peaks, _ = find_peaks(vol_norm, distance=min_distance, prominence=0.15)
es_peaks, _ = find_peaks(-vol_norm, distance=min_distance, prominence=0.15)

ed_peaks = np.array(sorted(ed_peaks))
es_peaks = np.array(sorted(es_peaks))

# =========================================================
# 5. PROPER ED → ES MATCHING (PHYSIOLOGICAL RULE)
# =========================================================
ef_list = []

for i in range(len(ed_peaks)):

    future_es = es_peaks[es_peaks > ed_peaks[i]]

    if len(future_es) == 0:
        continue

    es_idx = future_es[0]

    edv = vol_smooth[ed_peaks[i]]
    esv = vol_smooth[es_idx]

    if edv > 0:
        ef = ((edv - esv) / edv) * 100

        # physiological constraint
        if 20 <= ef <= 80:
            ef_list.append(ef)

ef_list = np.array(ef_list)

# =========================================================
# 6. CLINICAL REPORT
# =========================================================
mean_ef = np.mean(ef_list) if len(ef_list) > 0 else 0
std_ef = np.std(ef_list) if len(ef_list) > 0 else 0

print("\n================ CARDIAC ANALYSIS REPORT ================")
print("Total frames:", len(volumes))
print("Cycles detected:", len(ef_list))
print("Mean EF:", round(mean_ef, 2))
print("EF variability:", round(std_ef, 2))

# =========================================================
# 7. MEDICAL-GRADE INTERPRETATION
# =========================================================
print("\n================ CLINICAL INTERPRETATION ================")

if len(ef_list) < 2:
    print("⚠ Insufficient cycles for reliable assessment")

else:
    if mean_ef < 40:
        print("⚠ Reduced Left Ventricular Systolic Function (Low EF)")

    elif 40 <= mean_ef <= 55:
        if std_ef > 12:
            print("⚠ Borderline LV Function with variability")
        else:
            print("⚠ Mildly Reduced / Borderline LV Function")

    else:
        if std_ef > 12:
            print("⚠ Mild functional variability detected")
        else:
            print("✅ Normal Left Ventricular Function")

# =========================================================
# 8. VISUALIZATION (GUIDE DEMO READY)
# =========================================================
plt.figure(figsize=(10,4))
plt.plot(vol_smooth, label="LV Volume Curve (Smoothed)")

if len(ed_peaks) > 0:
    plt.scatter(ed_peaks, vol_smooth[ed_peaks], color="red", label="ED")

if len(es_peaks) > 0:
    plt.scatter(es_peaks, vol_smooth[es_peaks], color="blue", label="ES")

plt.title("Cardiac Cycle Analysis (Final Fixed Version)")
plt.xlabel("Frame")
plt.ylabel("LV Volume (pixel area)")
plt.legend()
plt.grid()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,4))
plt.plot(vol_smooth, label="LV Volume Curve", linewidth=2)

plt.scatter(ed_peaks, vol_smooth[ed_peaks], color='red', label="ED (max filling)")
plt.scatter(es_peaks, vol_smooth[es_peaks], color='blue', label="ES (max contraction)")

plt.title("Left Ventricle Volume Over Time")
plt.xlabel("Frame")
plt.ylabel("LV Volume (pixel area)")
plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(8,4))

plt.plot(ef_list, marker='o', linewidth=2)
plt.axhline(55, linestyle='--', color='green', label='Normal EF Reference')
plt.axhline(40, linestyle='--', color='red', label='Low EF Threshold')

plt.title("Ejection Fraction per Cardiac Cycle")
plt.xlabel("Cycle")
plt.ylabel("EF (%)")
plt.legend()
plt.grid()
plt.show()

In [ ]:

import matplotlib.pyplot as plt
import matplotlib.animation as animation
import torch
import numpy as np
from IPython.display import HTML

model.eval()

num_frames = min(30, len(val_ds))

fig, ax = plt.subplots(1, 3, figsize=(10,4))

def to_img(x):
    # 🔥 FIX: handle (C,H,W) properly
    if torch.is_tensor(x):
        x = x.detach().cpu().numpy()

    if x.ndim == 3:
        # if multi-channel → take first channel
        x = x[0]

    return x

def update(i):

    img, mask = val_ds[i]
    img_in = img.unsqueeze(0).to(device).float()

    with torch.no_grad():
        pred = model(img_in)[0][0].cpu().numpy()

    pred_bin = (pred > 0.5).astype(np.float32)

    img_np = to_img(img)
    mask_np = to_img(mask)

    ax[0].clear()
    ax[1].clear()
    ax[2].clear()

    ax[0].imshow(img_np, cmap='gray')
    ax[0].set_title("Input")
    ax[0].axis('off')

    ax[1].imshow(mask_np, cmap='gray')
    ax[1].set_title("Ground Truth")
    ax[1].axis('off')

    ax[2].imshow(img_np, cmap='gray')
    ax[2].imshow(pred_bin, cmap='jet', alpha=0.5)
    ax[2].set_title("Prediction")
    ax[2].axis('off')

ani = animation.FuncAnimation(fig, update, frames=num_frames, interval=200, blit=False)

plt.close(fig)

HTML(ani.to_jshtml())